Análise de Dados

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

Carregamento dos Dados

In [ ]:
conn = sqlite3.connect("data/publishers_magazines.db")
df_transactions = pd.read_sql_query("SELECT * FROM transactions", conn)
df_magazines = pd.read_sql_query("SELECT * FROM magazines", conn)
df_publishers = pd.read_sql_query("SELECT * FROM publishers", conn)
df_warehouses = pd.read_sql_query("SELECT * FROM warehouses", conn)
conn.close()

display(df_transactions.head())
display(df_magazines.head())
display(df_publishers.head())

Análise de Vendas por magazine

In [ ]:

df_tx_mag = df_transactions.merge(df_magazines, on="magazine_id", how="left")
sales_by_category = df_tx_mag.groupby("magazine_category")["amount"].sum().reset_index()
sales_by_category = sales_by_category.sort_values("amount", ascending=False)
display(sales_by_category)

Top 5 publishers

In [ ]:

df_tx_pub = df_transactions.merge(df_publishers, on="publisher_id", how="left")
sales_by_publisher = df_tx_pub.groupby("publisher_name")["amount"].sum().reset_index()
sales_by_publisher = sales_by_publisher.sort_values("amount", ascending=False)
top5_publishers = sales_by_publisher.head(5)
display(top5_publishers)

Gráfico- Vendas por categoria

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(sales_by_category["magazine_category"], sales_by_category["amount"], color="skyblue")
plt.title("Total de Vendas por Categoria de Revista")
plt.xlabel("Categoria")
plt.ylabel("Total de Vendas")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Gráfico- 5 maiores publishers

In [ ]:
plt.figure(figsize=(8, 8))
plt.pie(top5_publishers["amount"], labels=top5_publishers["publisher_name"], autopct="%1.1f%%", startangle=140)
plt.title("Distribuição das Vendas - Top 5 Editores")
plt.tight_layout()
plt.show()

Filtro de Transações

In [ ]:
def filtrar_transacoes(df_transactions, df_publishers, data_inicio=None, data_fim=None, publisher_name=None):
    """
    Filtra transações por intervalo de datas e/ou nome do editor.
    
    Args:
        df_transactions: DataFrame de transações
        df_publishers: DataFrame de editores
        data_inicio: string no formato YYYY/MM/DD ou None
        data_fim: string no formato YYYY/MM/DD ou None
        publisher_name: string com o nome do editor ou None
    
    Returns:
        DataFrame filtrado com informações do editor incluídas.
    """
    df = df_transactions.copy()
    
   
    df["transaction_date"] = pd.to_datetime(df["transaction_date"], format="%Y/%m/%d", errors="coerce")
    
    if data_inicio:
        df = df[df["transaction_date"] >= pd.to_datetime(data_inicio)]
    if data_fim:
        df = df[df["transaction_date"] <= pd.to_datetime(data_fim)]
    

    df = df.merge(df_publishers, on="publisher_id", how="left")
    
    if publisher_name:
        df = df[df["publisher_name"].str.contains(publisher_name, case=False, na=False)]
    
    return df


resultado = filtrar_transacoes(df_transactions, df_publishers, data_inicio="2022/01/01", data_fim="2022/12/31", publisher_name="Hawkins")
display(resultado)